In [0]:
base_path = "/Volumes/capstone_project/capstone_schema/capstone_volume"

In [0]:
%pip install dbldatagen

In [0]:
%restart_python

In [0]:
import dbldatagen as dg
from pyspark.sql.functions import rand, when, col
from datetime import timedelta, datetime

# --- 1. Parameters ---
BASE_ROW_COUNT = 4900000 
NUM_DAYS = 90

end_date = datetime.now()
start_date = end_date - timedelta(days=NUM_DAYS)

# --- 2. Build the Schema & Inject Flaws using dbldatagen ---
data_spec = (
    dg.DataGenerator(spark, name="ecommerce_orders_raw", rows=BASE_ROW_COUNT, partitions=8)
    
    # FLAW 1 (Null Handling): Inject 5% NULL values directly into order_id
    .withColumn("order_id", "string", 
                expr="concat('ORD-', lpad(cast(int(rand() * 99999999) as string), 8, '0'))", 
                percentNulls=0.05)
    
    # THE FIX: Bounded pools for realistic repeat purchases (15k customers, 500 products)
    .withColumn("customer_id", "string", 
                expr="concat('CUST-', lpad(cast(int(rand() * 15000) as string), 5, '0'))")
    .withColumn("product_id", "string", 
                expr="concat('PROD-', lpad(cast(int(rand() * 500) as string), 4, '0'))")
    
    # FLAW 2 (Category Normalization): Messy free-text categories
    .withColumn("category", "string", 
                values=["Electronics", "ELECTRONICS", "electronics ", "Elect.", 
                        "Apparel", "apparel", "APP", 
                        "Beauty", "beauty products",
                        "sports", "Sport", "Sports"])
    
    # FIXED: Added quantity back in
    .withColumn("quantity", "integer", minValue=-5, maxValue=10, random=True) 
    
    # FIXED: Placed unit_price on its own line
    .withColumn("unit_price", "float", expr="(rand() * 1550) - 50") 
    
    # FIXED: Only ONE discount column, using the high-variance expr method
    .withColumn("discount", "float", expr="rand() * 0.4", percentNulls=0.02)
    
    .withColumn("payment_method", "string", values=["Credit Card", "UPI", "Debit Card"], random=True)
    .withColumn("device_type", "string", values=["Mobile", "Desktop", "Tablet"], random=True)
    # FIXED: Using rock-solid Unix timestamp math to avoid Spark interval parsing errors.
    # 90 days * 24 hours * 60 minutes * 60 seconds = absolute uniform distribution!
    .withColumn("event_timestamp", "timestamp",
                expr=f"cast(from_unixtime(unix_timestamp('{start_date.strftime('%Y-%m-%d 00:00:00')}') + cast(rand() * {NUM_DAYS * 24 * 60 * 60} as bigint)) as timestamp)")
)

# Build the base DataFrame
print(f"Building base {BASE_ROW_COUNT} rows...")
df_base = data_spec.build()

# --- 3. FLAW 4 (Deduplication): Create Exact Duplicates ---
print("Injecting duplicate records for MERGE INTO testing...")
df_duplicates = df_base.sample(withReplacement=False, fraction=0.05)
df_dirty_final = df_base.union(df_duplicates)

# --- 4. Write to Cloud Storage ---
print(f"Writing 5 Million rows of messy data to {base_path}/input in JSON format...")
df_dirty_final.write.format("json").mode("overwrite").save(f"{base_path}/input")

print("Data generation complete! Your Bronze layer is ready for ingestion.")
display(df_dirty_final)

In [0]:
dataset = spark.read.format('json').load(f'{base_path}/input')

In [0]:
dataset.count()

In [0]:
dataset.printSchema()

In [0]:
from pyspark.sql.functions import col
dataset.filter(col("unit_price") < 0).count()

In [0]:
from pyspark.sql.functions import col
dataset.select(col('customer_id')).distinct().count()

In [0]:
from pyspark.sql import functions as F
dataset.select(F.max('customer_id'), F.min('customer_id')).show()

In [0]:
from pyspark.sql import functions as F

In [0]:
dataset.filter(F.col('discount') != 0).count()